# III-7 研究室の計算機を使う前に：空いている GPU・裏で動かす・ジョブ（ノートブック版）

教科書の III-7 のうち、**研究室のアカウントが無くても確かめられる部分**（手順 1〜5 と、手順 9 の GPU の絞り方の練習）を、上から順に実行すれば動く 1 冊にしたものです。Google Colab の GPU つきのランタイムは、NVIDIA の GPU と `nvidia-smi` のある Linux なので、研究室の GPU サーバーの小さな練習台になります。Mac の Jupyter でも動きます（GPU の見え方の節は、MPS の表示だけになります）。

| 節 | 内容 | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 何が使えるかを調べる | 8 | `lab_check.sh` |
| 2 | 空いている GPU の判定（見本の値・この機械の値） | 2・9 | `gpu_free.py` |
| 3 | `CUDA_VISIBLE_DEVICES` で GPU の見え方を変える、確認問題 2 | 2・9 | `show_devices.py` |
| 4 | `nohup` で裏で動かす | 4 | `run_bg.sh` |
| 5 | ジョブスクリプトを Slurm なしで動かす、配列ジョブをまねる | 5・10 | `job_seeds.sbatch`・`seed_stats.py` |
| 6 | 1 枚に絞って GPU-Util を記録する、確認問題 7 | 9 | `gpu_log.sh`（III-5） |
| 7 | 確認問題 3（配列ジョブの時間と GPU 時間） | ― | ― |

- 次の「準備」のセルたちが、III-2 の `train_toy.py`、III-4 の `repro.py`・`runlog.py`・`train_repro.py`・`configs/seed_small.yaml`・`seed_stats.py`、III-5 の `gpu_log.sh` と、この章の 5 ファイルを、このノートブックの隣に `%%writefile` で書き出します。**中身は教科書と 1 文字も変えていません**。
- `run_bg.sh`・`job_seeds.sbatch` などは中で `uv run --locked python` を呼ぶので、このノートブックでは **`bin/uv` という小さな代わり**（`uv run --locked python` を、このノートブックの python に置き換えるだけ）を作って PATH の先頭に足します。本物の uv と `uv.lock` で環境をそろえること（手順 8）は、ここでは確かめられません。
- Colab の GPU は 1 枚なので、「ほかの人が使っている GPU」は 2 節の見本の値（5 枚）で練習します。Colab には Slurm が無いので、配列ジョブは `SLURM_ARRAY_TASK_ID` を変えながら 1 本ずつ順に実行してまねます（`#SBATCH` の行は bash にとってはただのコメントです）。
- 執筆時の Mac（Apple シリコン、MPS）で、全部で 2〜3 分ほどです。学習は `configs/seed_small.yaml` のまま（教科書と同じ。1 本 MPS で 8 秒前後）です。GPU の無いランタイムでは 1 本 1.5〜3 分かかるので、5 節の本数を 1 本に減らします。
- **入れていないもの**：研究室の決まりを聞くこと（手順 6）、SSH の鍵と `~/.ssh/config`（手順 7）、`git bundle` での送り方（手順 8 の前半）、Slurm の `sbatch`・`squeue`・`sacct`、Docker・Apptainer、tmux（Colab のセルからは対話の操作ができないので、Mac のターミナルで試してください）。
- ⚠ 共有サーバーでしてはいけないこと（教科書の手順の後の箱）は、Colab で練習するときも同じ考え方です。研究データを研究室の外（個人の Drive・Colab を含む）にコピーしないでください。このノートブックが使うのは合成データだけです。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備

Colab では、メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU（例：T4）を選んでから、**最初のセルから**実行してください（項目の名前は公式ドキュメントで要確認）。GPU を選ばなくても全部のセルは動きます。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・pandas・scipy・matplotlib・pyyaml・tensorboard が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys, os
for mod, pkg in [("torch", "torch"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                 ("yaml", "pyyaml"), ("tensorboard", "tensorboard")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import torch, platform
print("Python", platform.python_version(), " torch", torch.__version__, " CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
os.makedirs("configs", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

`train_toy.py`（III-2 の 6 節）を書き出します。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

`repro.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile repro.py
#!/usr/bin/env python3
"""III-4：device の選択・乱数の種・決定的な計算の設定を1か所にまとめる（Mac の MPS／CPU、クラウドの CUDA で共通）
使い方（ほかのスクリプトから）:
  from repro import pick_device, set_seed, set_determinism, seed_worker, make_generator, state_sha256
  set_determinism(True)            # ← CUDA を使う前（最初）に呼ぶ
  dev = pick_device("auto")
  set_seed(3)
動作の確認:
  uv run python repro.py           # 選ばれた device と設定を表示する
"""
import hashlib
import os
import random

import numpy as np
import torch


def pick_device(name="auto"):
    """cuda → mps → cpu の順に探す（III-1 の bench_step.py と同じ順番）"""
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed):
    """Python・NumPy・PyTorch（CPU と全部の GPU）の乱数の種をそろえる"""
    random.seed(seed)
    np.random.seed(seed % 2**32)
    torch.manual_seed(seed)          # CPU と CUDA の両方に効く（MPS も torch.manual_seed で設定される）


def seed_worker(worker_id):
    """DataLoader の worker ごとに NumPy と random の種を決める（PyTorch の公式の書き方）"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed):
    """データの並び順を決める乱数。CPU で作るので、どの device でも同じ並び順になる"""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def set_determinism(deterministic=True, tf32=False):
    """決定的な計算の設定。CUDA の最初の計算より前に呼ぶ（CUBLAS_WORKSPACE_CONFIG は cuBLAS の初期化時に読まれる）
    deterministic=True : 同じ機械・同じ版・同じ種なら、何度実行しても同じ結果を目指す（少し遅くなる）
    deterministic=False: 速さ優先（cudnn.benchmark=True）。同じ種でも実行ごとに少しずれる
    tf32=False        : Ampere 以降の GPU の TF32（float32 の行列積・畳み込みを精度を落として速くする）を使わない
    返り値: 実際に設定した内容（run_record.json に残す）
    """
    info = {"deterministic": bool(deterministic), "tf32": bool(tf32)}
    if deterministic:
        # 公式（NVIDIA cuBLAS の Results Reproducibility）の値。:16:8 はメモリが少ないが遅いことがある
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        # MPS には決定的な実装が無い演算があるので、MPS では「エラー」ではなく「警告」にする
        mps = torch.backends.mps.is_available() and not torch.cuda.is_available()
        torch.use_deterministic_algorithms(True, warn_only=mps)
        info["warn_only"] = mps
    else:
        torch.backends.cudnn.benchmark = True
        torch.use_deterministic_algorithms(False)
    # TF32 の切り替え（新しい版では別の名前の設定もある。公式ドキュメントで要確認）
    torch.set_float32_matmul_precision("high" if tf32 else "highest")
    torch.backends.cuda.matmul.allow_tf32 = bool(tf32)
    torch.backends.cudnn.allow_tf32 = bool(tf32)
    info["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", "")
    info["cudnn_benchmark"] = torch.backends.cudnn.benchmark
    return info


def state_sha256(model):
    """モデルの重みのハッシュ（先頭16文字）。2回の実行がビット単位で同じかを調べる"""
    h = hashlib.sha256()
    for k, v in sorted(model.state_dict().items()):
        h.update(k.encode())
        h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()[:16]


if __name__ == "__main__":
    info = set_determinism(True)
    dev = pick_device()
    set_seed(0)
    print("device:", dev, "/ torch", torch.__version__, "/ CUDA（ビルド）", torch.version.cuda)
    print("設定:", info)
    x = torch.randn(3, generator=make_generator(0))
    print("make_generator(0) の最初の3個:", [round(v, 4) for v in x.tolist()], "（どの機械でも同じ値になるはず）")

`runlog.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile runlog.py
#!/usr/bin/env python3
"""III-4：実行の記録（コミットID・設定・環境・データ）を run_record.json に残す。2つの記録の違いを表示する
- 設定は YAML。--set a.b=値 で一部だけ上書きできる（Hydra の上書きの書き方をまねた最小限のもの）
- 記録しないもの：環境変数の値（決まった名前だけ）、ホスト名（個人名が入ることがある）、秘密の値
使い方:
  （train_repro.py などから）from runlog import load_config, start_run, finish_run
  uv run python runlog.py show runs/seed_mps/seed000            # 1つの記録を見る
  uv run python runlog.py diff runs/seed_mps/seed000 runs/seed_cuda/seed000   # 何が違うか
"""
import argparse
import copy
import datetime
import hashlib
import json
import os
import platform
import subprocess
import sys

import yaml

# 記録してよい環境変数（値に秘密が入らないものだけ）。HF_TOKEN や WANDB_API_KEY は「名前があるか」だけ記録する
ENV_KEEP = ["CUBLAS_WORKSPACE_CONFIG", "PYTHONHASHSEED", "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS",
            "UV_TORCH_BACKEND", "WANDB_MODE", "HF_HOME"]
ENV_SECRET_NAMES = ["HF_TOKEN", "WANDB_API_KEY", "VAST_API_KEY", "GH_TOKEN"]
CODE_SUFFIX = (".py", ".sh", ".yaml", ".yml", ".toml")   # コミットしていなければ「変更あり」とみなすファイル


# ---------------------------------------------------------------- 設定ファイル
def _parse_value(text):
    """'0.01' → 0.01、'true' → True、'[1,2]' → [1, 2]（YAML の書き方で読む）"""
    return yaml.safe_load(text)


def apply_overrides(cfg, overrides):
    """['train.epochs=5', 'model.width=32'] のような上書きを適用する。無い鍵はエラー（打ち間違いを防ぐ）"""
    cfg = copy.deepcopy(cfg)
    for item in overrides or []:
        if "=" not in item:
            raise SystemExit(f"--set の書き方が違います（鍵=値）: {item}")
        key, val = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            if not isinstance(node, dict) or p not in node:
                raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
            node = node[p]
        if not isinstance(node, dict) or parts[-1] not in node:
            raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
        node[parts[-1]] = _parse_value(val)
    return cfg


def load_config(path, overrides=None):
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return apply_overrides(cfg, overrides)


def config_hash(cfg):
    """設定の中身のハッシュ（鍵の順番や空白に左右されない）。同じ値なら同じ設定"""
    text = json.dumps(cfg, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# ---------------------------------------------------------------- Git・ファイル・環境
def _git(*args):
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else None
    except (OSError, subprocess.SubprocessError):
        return None


def git_info():
    commit = _git("rev-parse", "HEAD")
    if commit is None:
        return {"commit": "unknown", "dirty": None}
    lines = (_git("-c", "core.quotepath=off", "status", "--porcelain", "--untracked-files=all") or "").splitlines()
    changed = [x for x in lines if not x.startswith("??")]
    # Git に入っていない新しいコード（import されうる）も「変更あり」とみなす。data/ runs/ は .gitignore で見えない
    untracked_code = [x[3:].strip('"') for x in lines if x.startswith("??") and x.strip('"').endswith(CODE_SUFFIX)]
    return {"commit": commit, "short": commit[:7],
            "branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
            "dirty": bool(changed or untracked_code), "changed_files": changed,
            "untracked_code": untracked_code}


def file_sha256(path, n=12):
    if not os.path.exists(path):
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:n]


def env_info(device=None):
    info = {"python": sys.version.split()[0], "os": platform.system(), "os_release": platform.release(),
            "machine_arch": platform.machine(), "uv_lock_sha256": file_sha256("uv.lock")}
    try:
        import numpy
        info["numpy"] = numpy.__version__
    except ImportError:
        pass
    try:
        import torch
        info["torch"] = torch.__version__
        info["torch_cuda"] = torch.version.cuda
        info["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
        info["mps_available"] = torch.backends.mps.is_available()
        if device is not None and str(device).startswith("cuda"):
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    except ImportError:
        info["torch"] = None
    info["env"] = {k: os.environ[k] for k in ENV_KEEP if k in os.environ}
    info["secrets_present"] = [k for k in ENV_SECRET_NAMES if os.environ.get(k)]   # 名前だけ。値は残さない
    return info


# ---------------------------------------------------------------- 記録
def start_run(outdir, cfg, seed, machine, device=None, extra=None, allow_dirty=False):
    """実行の最初に呼ぶ。run_record.json を書き、コミットしていない変更があれば差分も残す"""
    os.makedirs(outdir, exist_ok=True)
    g = git_info()
    if g.get("dirty"):
        if not allow_dirty:
            raise SystemExit("コミットしていない変更があります。コミットしてから実行するか、"
                             "練習なら --allow-dirty を付けてください（差分を uncommitted.diff に残します）")
        with open(os.path.join(outdir, "uncommitted.diff"), "w", encoding="utf-8") as f:
            f.write(_git("diff", "HEAD") or "")
            for p in g.get("untracked_code", []):          # 新しいファイルは中身ごと残す
                f.write(f"\n# ===== Git に入っていないファイル: {p} =====\n")
                try:
                    with open(p, encoding="utf-8", errors="replace") as src:
                        f.write(src.read())
                except OSError:
                    f.write("（読めませんでした）\n")
    rec = {"started": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
           "argv": sys.argv, "machine": machine, "seed": seed,
           "git": g, "config": cfg, "config_hash": config_hash(cfg),
           "env": env_info(device), "extra": extra or {}}
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def finish_run(outdir, rec, results):
    rec = dict(rec)
    rec["finished"] = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
    rec["results"] = results
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)


# ---------------------------------------------------------------- 表示・比較
def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        else:
            out[key] = v
    return out


def load_record(path):
    if os.path.isdir(path):
        path = os.path.join(path, "run_record.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


IGNORE = ("started", "finished", "argv", "results.wall_s", "git.changed_files", "git.untracked_code")


def diff_records(a, b):
    fa, fb = flatten(a), flatten(b)
    rows = []
    for k in sorted(set(fa) | set(fb)):
        if k.startswith(IGNORE):
            continue
        if fa.get(k) != fb.get(k):
            rows.append((k, fa.get(k), fb.get(k)))
    return rows


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("show")
    s.add_argument("run")
    d = sub.add_parser("diff")
    d.add_argument("a")
    d.add_argument("b")
    a = ap.parse_args()
    if a.cmd == "show":
        r = load_record(a.run)
        for k, v in flatten(r).items():
            print(f"{k:32s} {v}")
    else:
        rows = diff_records(load_record(a.a), load_record(a.b))
        if not rows:
            print("違いはありません（時刻・コマンドラインを除く）")
        for k, va, vb in rows:
            print(f"{k:32s} {str(va):28s} | {vb}")
        keys = [k for k, _, _ in rows]
        important = [k for k in keys if k.startswith(("git.commit", "env.uv_lock", "config_hash", "env.torch",
                                                      "extra.data_id"))]
        if important:
            print("\n注意：コード・環境・設定・データのどれかが違います。結果の差がこれのせいか、"
                  "種や device のせいかを区別できません:", ", ".join(important))


if __name__ == "__main__":
    main()

`train_repro.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile train_repro.py
#!/usr/bin/env python3
"""III-4：種を1つ決めて小さな学習を1回行い、結果と「何で動かしたか」を全部残す
- データ（合成のモータ音）とモデルは III-2 の train_toy.py のものを使う
- 種を変えて何回も実行し（run_seeds.sh）、seed_stats.py で種のばらつき・MPS と CUDA の差を調べる
- 結果は OUT/seedNNN[_rK]/ に run_record.json・metrics.csv・final.json・tb/（TensorBoard）
使い方:
  uv run python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/seed_mps --machine mac
  uv run python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/seed_cpu --device cpu --rep 1
  uv run python train_repro.py ... --set train.epochs=5 --tracker none
"""
import argparse
import csv
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from repro import make_generator, pick_device, seed_worker, set_determinism, set_seed, state_sha256
from runlog import config_hash, finish_run, load_config, start_run, write_json
from train_toy import AudioCNN, SynthMotor


class NoisyMotor(Dataset):
    """train_toy.py の SynthMotor に雑音を足して、問題を難しくする（正解率が 1 に張り付かないように）。
    雑音も番号 i と種から決めるので、何度作っても同じデータになる"""
    def __init__(self, n, length, seed, noise):
        self.base, self.seed, self.noise = SynthMotor(n, length, seed), seed, noise

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, y = self.base[i]
        if self.noise > 0:
            rng = np.random.default_rng([self.seed, i, 7])
            x = x + torch.from_numpy(rng.normal(0, self.noise, x.shape[-1]).astype(np.float32))
        return x, y


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/seed_small.yaml")
    ap.add_argument("--seed", type=int, required=True, help="モデルの初期値とデータの並び順の種")
    ap.add_argument("--rep", type=int, default=0, help="同じ種のくり返しの番号（決定的かを調べる）")
    ap.add_argument("--out", required=True, help="例：runs/seed_mps")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--machine", default="mac", help="機械の名前（ホスト名は記録しないので自分で付ける）")
    ap.add_argument("--set", action="append", default=[], metavar="KEY=VALUE", help="設定の上書き")
    ap.add_argument("--tracker", choices=["tensorboard", "wandb", "none"], default="tensorboard")
    ap.add_argument("--allow-dirty", action="store_true", help="コミットしていない変更があっても実行する（練習用）")
    return ap.parse_args()


@torch.no_grad()
def accuracy(model, loader, dev):
    model.eval()
    correct = n = 0
    for x, y in loader:
        pred = model(x.to(dev)).argmax(1).cpu()
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def open_tracker(kind, outdir, cfg, name, machine, commit):
    """記録係を開く。返り値は log(dict, step) と close() を持つ小さな入れ物"""
    if kind == "tensorboard":
        from torch.utils.tensorboard import SummaryWriter
        w = SummaryWriter(log_dir=os.path.join(outdir, "tb"))

        class TB:
            def log(self, d, step):
                for k, v in d.items():
                    w.add_scalar(k, v, step)

            def close(self):
                w.flush()
                w.close()
        return TB()
    if kind == "wandb":
        # 既定は offline（キーをインスタンスに置かない）。Mac で wandb sync して送る
        os.environ.setdefault("WANDB_MODE", "offline")
        # W&B は既定でホスト名や Git の情報（リモートの URL・差分など）も送る。自分で付けた名前とコミットIDだけにする
        os.environ.setdefault("WANDB_HOST", machine)          # 実際のホスト名の代わり
        os.environ.setdefault("WANDB_DISABLE_GIT", "true")    # Git の情報を集めない（コミットIDは config に入れる）
        os.environ.setdefault("WANDB_DISABLE_CODE", "true")   # コードや差分を保存しない
        import wandb   # uv sync --group wandb で入れたときだけ使う
        run = wandb.init(project="home-cloud", name=name, config=dict(cfg, commit=commit), dir=outdir)

        class WB:
            def log(self, d, step):
                run.log(d, step=step)

            def close(self):
                run.finish()
        return WB()

    class Nothing:
        def log(self, d, step):
            pass

        def close(self):
            pass
    return Nothing()


def main():
    a = parse()
    cfg = load_config(a.config, a.set)
    rcfg = cfg["repro"]
    det = set_determinism(rcfg["deterministic"], rcfg["tf32"])   # CUDA を使う前に
    dev = pick_device(a.device)
    if rcfg.get("amp"):
        raise SystemExit("この章の比較では amp: false にします（configs/seed_small.yaml）")
    name = f"seed{a.seed:03d}" + (f"_r{a.rep}" if a.rep else "")
    outdir = os.path.join(a.out, name)
    final_path = os.path.join(outdir, "final.json")
    if os.path.exists(final_path):
        print(f"{final_path} があるので、とばします（やり直すならフォルダを消す）")
        return
    data_id = "synth-" + config_hash(cfg["data"])      # 本物のデータなら datahash.py の dataset_id を入れる
    rec = start_run(outdir, cfg, a.seed, a.machine, device=dev,
                    extra={"device": str(dev), "rep": a.rep, "determinism": det, "data_id": data_id},
                    allow_dirty=a.allow_dirty)
    print(f"{name}: device={dev} commit={rec['git'].get('short')} config={rec['config_hash']} "
          f"deterministic={det['deterministic']}", flush=True)

    d, m, t = cfg["data"], cfg["model"], cfg["train"]
    train_ds = NoisyMotor(d["n_train"], d["length"], d["seed"], d["noise"])
    val_ds = NoisyMotor(d["n_val"], d["length"], d["seed"] + 10_000, d["noise"])
    test_ds = NoisyMotor(d["n_test"], d["length"], d["seed"] + 20_000, d["noise"])
    kw = dict(num_workers=t["workers"], worker_init_fn=seed_worker)
    train_loader = DataLoader(train_ds, batch_size=t["batch"], shuffle=True,
                              generator=make_generator(a.seed), **kw)
    val_loader = DataLoader(val_ds, batch_size=256, **kw)
    test_loader = DataLoader(test_ds, batch_size=256, **kw)

    set_seed(a.seed)
    model = AudioCNN(w=m["width"])          # 初期値は CPU で作る → どの device でも同じ初期値
    init_hash = state_sha256(model)
    model = model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=t["lr"])
    # 学習率を最後に向けて0まで下げる（コサイン）。最後のエポックの値が落ち着き、種の比較がしやすくなる
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=t["epochs"] * len(train_loader))
    tracker = open_tracker(a.tracker, outdir, cfg, f"{os.path.basename(a.out)}/{name}",
                           a.machine, rec["git"].get("commit"))

    rows, step, t0 = [], 0, time.time()
    for epoch in range(t["epochs"]):
        loss_sum = n = 0
        for x, y in train_loader:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.item() * len(y)
            n += len(y)
            step += 1
            if step % 20 == 0:
                tracker.log({"train/loss": loss.item()}, step)
        val_acc = accuracy(model, val_loader, dev)
        row = {"epoch": epoch, "train_loss": loss_sum / n, "val_acc": val_acc, "elapsed_s": time.time() - t0}
        rows.append(row)
        tracker.log({"val/acc": val_acc, "train/epoch_loss": row["train_loss"]}, step)
        print(f"  epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}", flush=True)

    with open(os.path.join(outdir, "metrics.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    results = {"seed": a.seed, "rep": a.rep, "device": dev.type, "machine": a.machine,
               "val_acc": rows[-1]["val_acc"], "test_acc": accuracy(model, test_loader, dev),
               "final_loss": rows[-1]["train_loss"], "init_sha": init_hash,
               "state_sha": state_sha256(model), "wall_s": round(time.time() - t0, 2),
               "deterministic": det["deterministic"], "config_hash": rec["config_hash"],
               "commit": rec["git"].get("commit"), "uv_lock": rec["env"].get("uv_lock_sha256"),
               "torch": rec["env"].get("torch"), "data_id": data_id}
    tracker.close()
    finish_run(outdir, rec, results)
    write_json(final_path, results)            # 最後に書く（これがあれば「完了」）
    print(json.dumps({k: results[k] for k in ("val_acc", "test_acc", "state_sha", "wall_s")}), flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

`configs/seed_small.yaml`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile configs/seed_small.yaml
# III-4：種のばらつきを測るための小さな設定（Mac の MPS で1回 15〜40 秒程度の目安。機械と負荷で変わる）
# 値を変えるときは、このファイルを直してコミットするか、--set train.epochs=5 のように上書きする
name: seed_small
data:
  seed: 0          # 合成データの種。種のばらつきを測るときは固定する（変えるのはモデルの種だけ）
  n_train: 4000
  n_val: 1000
  n_test: 2000
  length: 4096
  noise: 1.5       # 足す雑音の標準偏差（信号の標準偏差＝1 に対して）。0 だと正解率がほぼ 1 に張り付く
model:
  width: 16
train:
  epochs: 3
  batch: 64
  lr: 0.0003       # 学習率の最大値（コサインで最後に 0 まで下げる）
  workers: 0       # 0 なら並び順が worker の数に左右されない。速さが要るときは 2〜4
repro:
  deterministic: true
  tf32: false      # MPS と CUDA を比べるので、どちらも float32 のまま計算する
  amp: false       # 混合精度は使わない（使うと CUDA だけ float16 が混じり、比較の条件がそろわない）

`seed_stats.py`（III-4 の 6 節）を書き出します。

In [ ]:
%%writefile seed_stats.py
#!/usr/bin/env python3
"""III-4：乱数の種によるばらつきを測り、MPS と CUDA の差がそのばらつきに比べて大きいかを調べる
- 種ごとの結果（final.json）から、平均の95%区間と「次の1つの種」の95%予測区間を出す
- 「良い種を選ぶ」誤用：最良の種の値がどのくらい楽観的か（n 個の最大値の期待値）
- 同じ種のくり返し（--rep）で、決定的になっているか（重みのハッシュが一致するか）
- 2つのグループ（例：MPS と CUDA）を種で対にして、差の検定と同等性の検定（TOST）
使い方:
  uv run --locked python seed_stats.py --demo     # 合成データ（runs/demo_seeds/）で動きを確かめる。図は runs/seed_stats.png
  uv run --locked python seed_stats.py runs/seed_mps runs/seed_cuda --metric test_acc --margin 0.01 --out runs/seed_compare.png
--margin（同等とみなす差の幅）は、結果を見る前に決めて、ノートに書いておく。
"""
import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import integrate, stats

plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False


# ---------------------------------------------------------------- 合成データ（--demo）
def make_demo(root, n_seeds=20, seed=10):
    """MPS と CUDA で同じ種 0〜19 を1回ずつ＋種0〜2 をもう1回（どちらも決定的：同じハッシュ）。
    真の値（seed_small.yaml を Mac の MPS と CPU で回した実測に近づけた）：平均 0.896、
    種の効果（初期値・並び順。device をまたいで共通）SD 0.012、device ごとに別れる部分 SD 0.0007、
    評価データの有限さ（種ごとに共通）val SD 0.008・test SD 0.005、CUDA − MPS の真の差 +0.0003"""
    rng = np.random.default_rng(seed)
    a = rng.normal(0, 0.012, n_seeds)
    ev = rng.normal(0, 0.008, n_seeds)
    et = rng.normal(0, 0.005, n_seeds)
    for dev, shift in (("mps", 0.0), ("cuda", 0.0003)):
        out = os.path.join(root, f"seed_{dev}")
        for s in range(n_seeds):
            g = 0.896 + shift + a[s] + rng.normal(0, 0.0007)
            val, test = g + ev[s], g + et[s]
            sha = f"{dev}{s:03d}"
            for rep in ((0, 1) if s < 3 else (0,)):
                d = os.path.join(out, f"seed{s:03d}" + (f"_r{rep}" if rep else ""))
                os.makedirs(d, exist_ok=True)
                res = {"seed": s, "rep": rep, "device": dev, "machine": "demo",
                       "val_acc": round(min(val, 1.0), 4), "test_acc": round(min(test, 1.0), 4),
                       "init_sha": f"init{s:03d}", "state_sha": sha, "deterministic": True,
                       "config_hash": "demo0config", "commit": "demo", "uv_lock": "demo",
                       "torch": "demo", "data_id": "demo"}
                with open(os.path.join(d, "final.json"), "w", encoding="utf-8") as f:
                    json.dump(res, f, ensure_ascii=False, indent=1)
    return [os.path.join(root, "seed_mps"), os.path.join(root, "seed_cuda")]


# ---------------------------------------------------------------- 読み込み
def load_runs(dirs):
    rows = []
    for d in dirs:
        for p in sorted(glob.glob(os.path.join(d, "seed*", "final.json"))):
            with open(p, encoding="utf-8") as f:
                r = json.load(f)
            r["group"] = os.path.basename(os.path.normpath(d))
            rows.append(r)
    if not rows:
        raise SystemExit("final.json が見つかりません: " + " ".join(dirs))
    return pd.DataFrame(rows)


def check_conditions(df):
    """比べてよい条件か（設定・データ・環境）を確かめる"""
    print("== 条件の確認 ==")
    for col in ("config_hash", "data_id", "commit", "uv_lock", "torch"):
        if col not in df:
            continue
        if col == "torch":   # 2.14.1 と 2.14.1+cu126 は同じ版（+ のあとは CUDA の置き場所の印）
            df = df.assign(torch=df["torch"].astype(str).str.split("+").str[0])
        per = df.groupby("group")[col].nunique()
        if (per > 1).any():
            print(f"  注意：同じグループの中で {col} が混ざっています: {per[per > 1].to_dict()}")
        vals = df.groupby("group")[col].first().to_dict()
        same = len(set(vals.values())) == 1
        mark = "同じ" if same else "違う"
        print(f"  {col:12s}: {mark}  {vals}")
        if not same and col in ("config_hash", "data_id"):
            print("    → 設定かデータが違うので、グループの差を device の差とは言えません")


# ---------------------------------------------------------------- 統計
def expected_max_z(n):
    """標準正規分布から n 個とったときの最大値の期待値 E[max Z]"""
    f = lambda x: x * n * stats.norm.pdf(x) * stats.norm.cdf(x) ** (n - 1)
    return integrate.quad(f, -10, 10)[0]


def summarize(y, label):
    n, m, s = len(y), y.mean(), y.std(ddof=1)
    tq = stats.t.ppf(0.975, n - 1)
    ci = (m - tq * s / np.sqrt(n), m + tq * s / np.sqrt(n))
    pi = (m - tq * s * np.sqrt(1 + 1 / n), m + tq * s * np.sqrt(1 + 1 / n))
    print(f"  {label}: n={n}  平均 {m:.4f}  SD {s:.4f}  最小 {y.min():.4f}  最大 {y.max():.4f}")
    print(f"    平均の95%区間        [{ci[0]:.4f}, {ci[1]:.4f}]（幅 ±{tq * s / np.sqrt(n):.4f}）")
    print(f"    次の1つの種の95%予測区間 [{pi[0]:.4f}, {pi[1]:.4f}]（幅 ±{tq * s * np.sqrt(1 + 1 / n):.4f}）")
    return {"n": n, "mean": m, "sd": s, "ci": ci, "pi": pi}


def best_seed_misuse(g, metric):
    """誤用の例：val で一番良い種を選び、その種の値を「この手法の性能」として報告する"""
    n = len(g)
    best = g.loc[g["val_acc"].idxmax()]
    em = expected_max_z(n)
    s = g[metric].std(ddof=1)
    print(f"  誤用の例「良い種を選ぶ」：n={n} 個の種の最大値は、平均より約 {em:.2f}×SD = {em * s:.4f} 高く出る（期待値）")
    print(f"    val が最良の種 {int(best['seed'])}：val_acc {best['val_acc']:.4f}（val の平均 {g['val_acc'].mean():.4f}）"
          f" → その種の test_acc {best['test_acc']:.4f}（test の平均 {g['test_acc'].mean():.4f}）")
    print("    選んだ val の値は楽観的。test では平均の方へ戻る（平均への回帰）。報告するのは全部の種の平均と区間")
    return em


def determinism_check(df):
    print("== 決定的か（同じ種のくり返しで、重みのハッシュが一致するか） ==")
    any_rep = False
    for grp, g in df.groupby("group"):
        multi = g.groupby("seed").filter(lambda x: len(x) > 1)
        if multi.empty:
            continue
        any_rep = True
        same = multi.groupby("seed")["state_sha"].nunique().eq(1)
        spread = multi.groupby("seed")["test_acc"].agg(lambda x: x.max() - x.min())
        print(f"  {grp}: くり返した種 {len(same)} 個のうち、ハッシュが一致 {int(same.sum())} 個。"
              f"test_acc の差の最大 {spread.max():.4f}")
    if not any_rep:
        print("  （くり返しがありません。--rep 1 で同じ種をもう一度実行すると調べられます）")


def paired_compare(a, b, metric, margin, alpha=0.05):
    """b − a を種で対にして比べる。TOST：差の 90% 区間が (−margin, +margin) に入れば同等"""
    m = a[["seed", metric]].merge(b[["seed", metric]], on="seed", suffixes=("_a", "_b"))
    n = len(m)
    if "init_sha" in a and "init_sha" in b:
        ii = a[["seed", "init_sha"]].merge(b[["seed", "init_sha"]], on="seed")
        same = int((ii["init_sha_x"] == ii["init_sha_y"]).sum())
        print(f"  初期値の重みのハッシュが2つのグループで一致した種：{same}/{len(ii)}（一致しなければ種の使い方が違う）")
    if n < 3:
        raise SystemExit("対にできる種が3個未満です")
    d = (m[f"{metric}_b"] - m[f"{metric}_a"]).to_numpy()
    dbar, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    df_ = n - 1
    t0 = dbar / se
    p_diff = 2 * stats.t.sf(abs(t0), df_)
    ci95 = (dbar - stats.t.ppf(0.975, df_) * se, dbar + stats.t.ppf(0.975, df_) * se)
    ci90 = (dbar - stats.t.ppf(1 - alpha, df_) * se, dbar + stats.t.ppf(1 - alpha, df_) * se)
    p_low = stats.t.sf((dbar + margin) / se, df_)        # H0: 差 ≤ −margin
    p_up = stats.t.cdf((dbar - margin) / se, df_)        # H0: 差 ≥ +margin
    p_tost = max(p_low, p_up)
    r = np.corrcoef(m[f"{metric}_a"], m[f"{metric}_b"])[0, 1]
    welch = stats.ttest_ind(m[f"{metric}_b"], m[f"{metric}_a"], equal_var=False)
    s_seed = np.sqrt((m[f"{metric}_a"].var(ddof=1) + m[f"{metric}_b"].var(ddof=1)) / 2)
    print(f"== 種で対にした比較（b − a、n={n}、指標 {metric}） ==")
    print(f"  差の平均 {dbar:+.4f}  差の SD {sd:.4f}  標準誤差 {se:.4f}  （種のばらつきの SD {s_seed:.4f}）")
    print(f"  差の検定（対応のある t）：t={t0:.2f}, 自由度 {df_}, p={p_diff:.3f}, 95%区間 [{ci95[0]:+.4f}, {ci95[1]:+.4f}]")
    print(f"  同等性の検定（TOST、幅 ±{margin}）：90%区間 [{ci90[0]:+.4f}, {ci90[1]:+.4f}], p={p_tost:.4f}"
          f" → {'同等と言える' if p_tost < alpha else '同等とは言えない（データが足りないか、差がある）'}")
    print(f"  参考：対にしない Welch の t 検定 p={welch.pvalue:.3f}。対の相関 r={r:.2f}"
          f"（種の効果が device をまたいで残る割合の目安）")
    # 対を無視して計算すると（別々の種で回したのと同じ扱い）：種のばらつきがそのまま差の誤差になる
    ya, yb = m[f"{metric}_a"].to_numpy(), m[f"{metric}_b"].to_numpy()
    va, vb = ya.var(ddof=1) / n, yb.var(ddof=1) / n
    se_u = np.sqrt(va + vb)
    df_u = (va + vb) ** 2 / (va ** 2 / (n - 1) + vb ** 2 / (n - 1))
    du = yb.mean() - ya.mean()
    tq = stats.t.ppf(1 - alpha, df_u)
    p_tost_u = max(stats.t.sf((du + margin) / se_u, df_u), stats.t.cdf((du - margin) / se_u, df_u))
    print(f"  参考：対を無視して計算すると（Welch の TOST）：90%区間 [{du - tq * se_u:+.4f}, {du + tq * se_u:+.4f}],"
          f" p={p_tost_u:.4f} → {'同等と言える' if p_tost_u < alpha else '同等とは言えない'}"
          f"（標準誤差 {se_u:.4f}。対にすると {se:.4f}）")
    if p_diff >= alpha and p_tost >= alpha:
        print("  誤用の例：「差の検定で p≥0.05 だから同じ」は誤り。同等と言うには TOST が要る（今回は言えない）")
    if p_diff < alpha and p_tost < alpha:
        print("  差は「ある」が、幅 ±margin より小さい（統計的に有意でも、実質的には同等）。どちらを言いたいかで検定を選ぶ")
    # TOST に必要な種の数の目安（真の差 0、検出力 80%）
    need = None
    for k in range(3, 1001):
        q = stats.t.ppf(1 - alpha, k - 1) + stats.t.ppf(1 - 0.2 / 2, k - 1)
        if k >= (q * sd / margin) ** 2:
            need = k
            break
    print(f"  目安：真の差が 0 のとき、この幅で80%の確率で同等と言うには、種が約 {need} 個要る（差の SD {sd:.4f} から）")
    return {"d": d, "dbar": dbar, "ci90": ci90, "ci95": ci95, "p_tost": p_tost, "p_diff": p_diff,
            "merged": m, "r": r, "need": need}


# ---------------------------------------------------------------- 図
def plot(df, groups, metric, res, margin, ems, out):
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    x0 = ax[0, 0]
    for i, grp in enumerate(groups):
        g = df[(df["group"] == grp) & (df["rep"] == 0)][metric].to_numpy()
        jit = np.random.default_rng(0).uniform(-0.12, 0.12, len(g))
        x0.plot(i + jit, g, "o", alpha=0.6)
        s = summarize_quiet(g)
        x0.errorbar(i + 0.3, s["mean"], yerr=[[s["mean"] - s["ci"][0]], [s["ci"][1] - s["mean"]]],
                    fmt="s", color="k", capsize=4)
        x0.errorbar(i + 0.4, s["mean"], yerr=[[s["mean"] - s["pi"][0]], [s["pi"][1] - s["mean"]]],
                    fmt="none", color="gray", capsize=4)
    x0.set_xticks(range(len(groups)))
    x0.set_xticklabels(groups)
    x0.set_ylabel(metric)
    x0.set_title("種ごとの結果（黒：平均の95%区間、灰：予測区間）")
    x1 = ax[0, 1]
    if res is not None:
        m = res["merged"]
        for _, r in m.iterrows():
            x1.plot([0, 1], [r[f"{metric}_a"], r[f"{metric}_b"]], "-o", color="tab:blue", alpha=0.4)
        x1.set_xticks([0, 1])
        x1.set_xticklabels(groups[:2])
        x1.set_title(f"同じ種を線で結ぶ（相関 r={res['r']:.2f}）")
    x2 = ax[1, 0]
    if res is not None:
        x2.axvspan(-margin, margin, color="tab:green", alpha=0.15, label=f"同等とみなす幅 ±{margin}")
        x2.errorbar(res["dbar"], 1, xerr=[[res["dbar"] - res["ci90"][0]], [res["ci90"][1] - res["dbar"]]],
                    fmt="o", capsize=5, label="差の90%区間（TOST）")
        x2.errorbar(res["dbar"], 0.6, xerr=[[res["dbar"] - res["ci95"][0]], [res["ci95"][1] - res["dbar"]]],
                    fmt="s", color="gray", capsize=5, label="差の95%区間")
        x2.plot(res["d"], np.full(len(res["d"]), 0.25), "|", color="k", ms=12, label="種ごとの差")
        x2.axvline(0, color="k", lw=0.8)
        x2.set_ylim(0, 1.4)
        x2.set_yticks([])
        x2.set_xlabel(f"{groups[1]} − {groups[0]}")
        x2.legend(fontsize=8, loc="upper left")
        x2.set_title("差と同等性の幅")
    x3 = ax[1, 1]
    ns = np.arange(1, 51)
    x3.plot(ns, [expected_max_z(k) for k in ns])
    for n_, e in ems.items():
        x3.plot(n_, e, "o", color="tab:red")
    x3.set_xlabel("試した種の数 n")
    x3.set_ylabel("最良の種の上ぶれ（SD の何倍か）")
    x3.set_title("「良い種を選ぶ」と楽観的になる量 E[max Z]")
    x3.grid(alpha=0.3)
    fig.tight_layout()
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    fig.savefig(out, dpi=120)
    print(f"図を保存しました: {out}")


def summarize_quiet(y):
    n, m, s = len(y), y.mean(), y.std(ddof=1)
    tq = stats.t.ppf(0.975, n - 1)
    return {"mean": m, "ci": (m - tq * s / np.sqrt(n), m + tq * s / np.sqrt(n)),
            "pi": (m - tq * s * np.sqrt(1 + 1 / n), m + tq * s * np.sqrt(1 + 1 / n))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dirs", nargs="*", help="runs/seed_mps runs/seed_cuda のように（2つ目 − 1つ目 を比べる）")
    ap.add_argument("--demo", action="store_true")
    ap.add_argument("--metric", default="test_acc")
    ap.add_argument("--margin", type=float, default=0.01, help="同等とみなす差の幅（結果を見る前に決める）")
    ap.add_argument("--out", default="runs/seed_stats.png", help="図の保存先（runs/ の中なら Git に入らない）")
    a = ap.parse_args()
    dirs = make_demo("runs/demo_seeds") if a.demo else a.dirs
    if not dirs:
        ap.error("フォルダを指定するか --demo を付けてください")
    df = load_runs(dirs)
    groups = [os.path.basename(os.path.normpath(d)) for d in dirs]
    check_conditions(df)
    print(f"== 種のばらつき（指標 {a.metric}、くり返しの rep=0 だけ使う） ==")
    ems = {}
    for grp in groups:
        g = df[(df["group"] == grp) & (df["rep"] == 0)]
        if len(g) < 2:
            print(f"  {grp}: 種が2個未満なので区間を出せません")
            continue
        summarize(g[a.metric].to_numpy(), grp)
        ems[len(g)] = best_seed_misuse(g, a.metric)
    determinism_check(df)
    res = None
    if len(groups) >= 2:
        res = paired_compare(df[(df["group"] == groups[0]) & (df["rep"] == 0)],
                             df[(df["group"] == groups[1]) & (df["rep"] == 0)], a.metric, a.margin)
    plot(df, groups, a.metric, res, a.margin, ems, a.out)


if __name__ == "__main__":
    main()

`gpu_log.sh`（III-5 の 6 節）を書き出します。

In [ ]:
%%writefile gpu_log.sh
#!/usr/bin/env bash
# III-5：コマンドを実行している間、nvidia-smi で GPU の状態を1秒ごとに CSV に記録する
# 使い方: bash gpu_log.sh <名前> -- <実行するコマンド…>
#   例: bash gpu_log.sh synth_w0 -- uv run --locked python train_fast.py --data synth --workers 0 --label synth_w0
# 出力: ${GPU_LOG_DIR:-runs/gpu_log}/<名前>.csv（nvidia-smi が無い Mac では記録せず、コマンドだけ実行する）
# nvidia-smi は CUDA_VISIBLE_DEVICES を見ずに全部の GPU を表示する。共有サーバー（III-7）で
# ほかの人の GPU が混ざらないよう、CUDA_VISIBLE_DEVICES があればその GPU だけを -i で記録し、
# まとめも GPU の番号（index）ごとに出す
set -euo pipefail
name="${1:?名前を指定してください}"
shift
[ "${1:-}" = "--" ] && shift
[ $# -gt 0 ] || { echo "実行するコマンドを -- のあとに書いてください" >&2; exit 2; }

if ! command -v nvidia-smi >/dev/null 2>&1; then
  echo "nvidia-smi がありません（Mac など）。GPU の記録はせずにコマンドだけ実行します。" >&2
  exec "$@"
fi

dir="${GPU_LOG_DIR:-runs/gpu_log}"
mkdir -p "${dir}"
out="${dir}/${name}.csv"
fields="timestamp,index,name,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,power.limit,temperature.gpu,clocks.sm"

# 記録する GPU を選ぶ（CUDA_VISIBLE_DEVICES が無い・空なら全部）
sel=()
cvd="${CUDA_VISIBLE_DEVICES:-}"
if [ -n "${cvd}" ]; then
  if nvidia-smi -i "${cvd}" -L >/dev/null 2>&1; then
    sel=(-i "${cvd}")
    echo "記録する GPU：CUDA_VISIBLE_DEVICES=${cvd}（nvidia-smi の番号として -i に渡す）" >&2
    case "${cvd}" in
      *GPU-*|*MIG-*) ;;                       # UUID なら並び順の問題は無い
      *) [ "${CUDA_DEVICE_ORDER:-}" = "PCI_BUS_ID" ] ||
           echo "注意：CUDA_DEVICE_ORDER=PCI_BUS_ID が無いと、CUDA の番号と nvidia-smi の番号がずれることがあります" >&2 ;;
    esac
  else
    echo "注意：nvidia-smi -i ${cvd} が通らないので全部の GPU を記録します。まとめの GPU ごとの行から自分の GPU を選んでください" >&2
  fi
fi
: > "${out}"                                   # 前の記録を先に消しておく
nvidia-smi "${sel[@]+"${sel[@]}"}" --query-gpu="${fields}" --format=csv,nounits -l 1 > "${out}" &
logger=$!
trap 'kill "${logger}" 2>/dev/null || true' EXIT

status=0
"$@" || status=$?

kill "${logger}" 2>/dev/null || true
wait "${logger}" 2>/dev/null || true
trap - EXIT

# まとめ：GPU ごとに、GPU-Util の平均と、50% 未満だった秒の割合、メモリと電力の最大
awk -F', *' 'NR > 1 && $4 ~ /^[0-9.]+$/ {
  g = $2; if (!(g in n)) order[++ng] = g
  n[g]++; u[g] += $4; if ($4 < 50) low[g]++
  if ($6 > m[g]) m[g] = $6; tot[g] = $7
  if ($8 ~ /^[0-9.]+$/ && $8 > p[g]) p[g] = $8; lim[g] = $9
  if ($10 > tc[g]) tc[g] = $10
} END {
  if (ng == 0) { print "記録がありません"; exit }
  for (k = 1; k <= ng; k++) {
    g = order[k]
    printf "GPU %s：記録 %d 秒、GPU-Util 平均 %.0f%%、50%% 未満の秒 %.0f%%、メモリ最大 %d / %d MiB、電力最大 %.0f / %s W、温度最大 %d ℃\n",
           g, n[g], u[g] / n[g], 100 * low[g] / n[g], m[g], tot[g], p[g], lim[g], tc[g]
  }
}' "${out}"
echo "記録：${out}"
exit "${status}"

`gpu_free.py`（III-7 の 6 節）を書き出します。

In [ ]:
%%writefile gpu_free.py
#!/usr/bin/env python3
"""III-7：共有の GPU サーバーで「いま空いている GPU」を調べ、CUDA_VISIBLE_DEVICES の候補を出す
- nvidia-smi の --query-gpu（メモリ・GPU-Util）と --query-compute-apps（動いているプロセス）を読む
- 「空き」＝ 使用メモリが --max-used-mib 以下 かつ GPU-Util が --max-util 以下 かつ プロセスが無い
  （値が [N/A] などで読めない GPU は、空きとみなさない）
- 研究室に予約表・Slurm があるときは、そちらが優先（このスクリプトは目安を出すだけ）
- nvidia-smi が無い Mac では、--demo で見本の表示を読んで動きを確かめられる
使い方:
  python3 gpu_free.py                 # 研究室のサーバーで
  python3 gpu_free.py --need 1        # 空いている GPU を1枚選んで export の行を表示
  python3 gpu_free.py --demo          # Mac で動作確認（見本の値）
"""
import argparse
import shutil
import subprocess
import sys

DEMO_GPU = """0, GPU-aaaa0000, 23510, 24564, 98
1, GPU-bbbb1111, 3, 24564, 0
2, GPU-cccc2222, 812, 24564, 0
3, GPU-dddd3333, 5, 24564, 0
4, GPU-eeee4444, [N/A], 24564, [Not Supported]"""
DEMO_APPS = """GPU-aaaa0000, 41233, 23490
GPU-cccc2222, 50871, 800"""


def run_smi(args):
    out = subprocess.run(["nvidia-smi", *args, "--format=csv,noheader,nounits"],
                         capture_output=True, text=True, check=True)
    return out.stdout.strip()


def num(text):
    """数値にできない値（[N/A]・[Not Supported] など）は nan にする（判定では使用中扱い）"""
    try:
        return float(text)
    except ValueError:
        return float("nan")


def parse(gpu_text, apps_text):
    gpus = []
    for line in gpu_text.splitlines():
        if not line.strip():
            continue
        idx, uuid, used, total, util = [s.strip() for s in line.split(",")]
        gpus.append({"index": int(idx), "uuid": uuid, "used": num(used),
                     "total": num(total), "util": num(util), "procs": 0})
    by_uuid = {g["uuid"]: g for g in gpus}
    for line in apps_text.splitlines():
        if not line.strip():
            continue
        uuid = line.split(",")[0].strip()
        if uuid in by_uuid:
            by_uuid[uuid]["procs"] += 1
    return gpus


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--max-used-mib", type=float, default=500)
    ap.add_argument("--max-util", type=float, default=5)
    ap.add_argument("--need", type=int, default=0, help="選ぶ枚数（0なら表だけ）")
    ap.add_argument("--demo", action="store_true")
    a = ap.parse_args()

    if a.demo:
        gpu_text, apps_text = DEMO_GPU, DEMO_APPS
        print("（見本の値で表示しています）")
    else:
        if shutil.which("nvidia-smi") is None:
            sys.exit("nvidia-smi がありません（Mac など）。--demo で動きを確かめられます。")
        gpu_text = run_smi(["--query-gpu=index,uuid,memory.used,memory.total,utilization.gpu"])
        apps_text = run_smi(["--query-compute-apps=gpu_uuid,pid,used_memory"])

    gpus = parse(gpu_text, apps_text)
    free = []
    print(f"{'GPU':>3}  {'使用メモリ MiB':>15}  {'Util%':>5}  {'プロセス':>4}  判定")
    for g in gpus:
        ok = g["used"] <= a.max_used_mib and g["util"] <= a.max_util and g["procs"] == 0
        if ok:
            free.append(g["index"])
        mark = "空き" if ok else "使用中"
        if g["used"] != g["used"] or g["util"] != g["util"]:   # nan（読めない値）
            mark = "不明（使用中扱い）"
        print(f"{g['index']:>3}  {g['used']:>7.0f} / {g['total']:<6.0f}  {g['util']:>5.0f}  {g['procs']:>8}  {mark}")
    print(f"空いている GPU：{free if free else 'なし'}")
    if a.need > 0:
        if len(free) < a.need:
            sys.exit(f"空きが {a.need} 枚ありません。待つか、研究室の予約表・ジョブスケジューラを使ってください。")
        pick = ",".join(str(i) for i in free[:a.need])
        print("次の2行を実行してから学習を始めます（番号は nvidia-smi と同じ並びにする）:")
        print("export CUDA_DEVICE_ORDER=PCI_BUS_ID")
        print(f"export CUDA_VISIBLE_DEVICES={pick}")


if __name__ == "__main__":
    main()

`show_devices.py`（III-7 の 6 節）を書き出します。

In [ ]:
%%writefile show_devices.py
#!/usr/bin/env python3
"""III-7：このプロセスから「見えている」GPU を表示する（CUDA_VISIBLE_DEVICES の確認）
- CUDA_VISIBLE_DEVICES=2 で起動すると、プロセスの中では cuda:0 が物理的な GPU 2 になる
- Slurm で GPU を頼んだジョブでも、Slurm が CUDA_VISIBLE_DEVICES を設定する
- Mac（MPS）・CPU だけの環境でも落ちずに表示する
使い方:
  uv run --locked python show_devices.py
  CUDA_VISIBLE_DEVICES=1 uv run --locked python show_devices.py
"""
import os

import torch


def main():
    for key in ("CUDA_VISIBLE_DEVICES", "CUDA_DEVICE_ORDER", "SLURM_JOB_ID"):
        print(f"{key} = {os.environ.get(key, '（未設定）')}")
    print(f"torch {torch.__version__}")
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print(f"CUDA：見えている GPU {n} 枚")
        for i in range(n):
            p = torch.cuda.get_device_properties(i)
            print(f"  cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
        print("cuda:0 が、CUDA_VISIBLE_DEVICES の最初に書いた GPU です。")
    elif torch.backends.mps.is_available():
        print("CUDA なし。MPS（Mac の GPU）が使えます。CUDA_VISIBLE_DEVICES は MPS には効きません。")
    else:
        print("CUDA も MPS もありません（CPU だけ）。CUDA_VISIBLE_DEVICES が空文字列だと GPU は1枚も見えません。")


if __name__ == "__main__":
    main()

`run_bg.sh`（III-7 の 6 節）を書き出します。

In [ ]:
%%writefile run_bg.sh
#!/usr/bin/env bash
# III-7：ログアウトしても止まらないように、コマンドを nohup で裏で実行する
# 使い方: bash run_bg.sh <名前> -- <コマンド…>
#   例: bash run_bg.sh seed000 -- uv run --locked python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/lab_seed --machine lab
# 出力: runs/logs/<名前>.log（標準出力とエラー）、runs/logs/<名前>.pid（プロセス番号）
# 様子を見る: tail -f runs/logs/<名前>.log    止める: kill "$(cat runs/logs/<名前>.pid)"
set -euo pipefail
name="${1:?名前を指定してください}"
shift
[ "${1:-}" = "--" ] && shift
[ $# -gt 0 ] || { echo "実行するコマンドを -- のあとに書いてください" >&2; exit 2; }

mkdir -p runs/logs
log="runs/logs/${name}.log"
pidf="runs/logs/${name}.pid"
if [ -f "${pidf}" ] && kill -0 "$(cat "${pidf}")" 2>/dev/null; then
  echo "${name} はもう動いています（PID $(cat "${pidf}")）。" >&2
  exit 1
fi

{
  echo "# 開始 $(date '+%Y-%m-%d %H:%M:%S')"
  echo "# CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-（未設定）}"
  echo "# コマンド: $*"
} > "${log}"
nohup "$@" >> "${log}" 2>&1 < /dev/null &
echo $! > "${pidf}"
echo "開始しました：PID $(cat "${pidf}")、ログ ${log}"
echo "様子を見る：tail -f ${log}"

`job_seeds.sbatch`（III-7 の 6 節）を書き出します。

In [ ]:
%%writefile job_seeds.sbatch
#!/usr/bin/env bash
#SBATCH --job-name=labuser-seeds
#SBATCH --array=0-9%2
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=02:00:00
#SBATCH --output=runs/logs/%x_%A_%a.out
# ↑ #SBATCH の行は Slurm だけが読む（bash にとってはただのコメント）。
#   パーティション（--partition）や GPU の書き方（--gres=gpu:1 か --gpus=1 か）、
#   時間・メモリの上限は研究室のクラスタの決まりで要確認。
#
# III-7：III-4 の種のくり返しを、Slurm の配列ジョブで1つの種ずつ実行する
# 使い方（Slurm）: mkdir -p runs/logs && sbatch job_seeds.sbatch
# 使い方（Slurm が無いサーバー・Mac で1つだけ試す）:
#   SLURM_ARRAY_TASK_ID=0 bash job_seeds.sbatch
#   DRY=1 SLURM_ARRAY_TASK_ID=3 bash job_seeds.sbatch   # 実行せずにコマンドだけ表示
# 環境変数: CONFIG（設定の YAML）・OUT_ROOT（既定 runs/lab_seed）・MACHINE（既定 lab。ホスト名は書かない）
#           WORKERS（設定したときだけ train.workers を上書きする。上書きすると config_hash が変わるので、
#           比べる実行どうし（Mac と研究室など）で同じ値にそろえる。既定は YAML のまま＝workers 0）
set -euo pipefail

# sbatch はジョブを出したフォルダ（SLURM_SUBMIT_DIR）で始まる。手で実行したときは今のフォルダ（~/home-cloud で実行する）
cd "${SLURM_SUBMIT_DIR:-.}"

task="${SLURM_ARRAY_TASK_ID:-0}"
seed=$(( task ))
config="${CONFIG:-configs/seed_small.yaml}"
out_root="${OUT_ROOT:-runs/lab_seed}"
run_dir="$(printf '%s/seed%03d' "${out_root}" "${seed}")"   # train_repro.py が作るフォルダ
mkdir -p runs/logs "${out_root}"

echo "# ジョブ ${SLURM_JOB_ID:-local} 配列 ${SLURM_ARRAY_JOB_ID:-local}_${task} 種 ${seed}"
echo "# CUDA_VISIBLE_DEVICES=${CUDA_VISIBLE_DEVICES:-（未設定）} CPU=${SLURM_CPUS_PER_TASK:-?}"
echo "# 開始 $(date '+%Y-%m-%d %H:%M:%S')"

if [ -f "${run_dir}/final.json" ]; then
  echo "${run_dir}/final.json があるので、とばします（III-4 と同じ）。"
  exit 0
fi

cmd=(uv run --locked python train_repro.py --config "${config}" --seed "${seed}" \
     --out "${out_root}" --machine "${MACHINE:-lab}")
if [ -n "${WORKERS:-}" ]; then
  cmd+=(--set "train.workers=${WORKERS}")      # --cpus-per-task 以下にする
fi
if [ "${DRY:-0}" = "1" ]; then
  printf '%q ' "${cmd[@]}"; echo
  exit 0
fi
"${cmd[@]}"
echo "# 終了 $(date '+%Y-%m-%d %H:%M:%S')"

`lab_check.sh`（III-7 の 6 節）を書き出します。

In [ ]:
%%writefile lab_check.sh
#!/usr/bin/env bash
# III-7：研究室のサーバーに初めて入った日に、何が使えるかを調べて表示する（何も変えない・何も入れない）
# 使い方: bash lab_check.sh
# 表示を研究室の外（チャット、個人のメモアプリ）に貼らない。ホスト名・ユーザー名は表示しない。
set -uo pipefail

line() { printf '%-22s %s\n' "$1" "$2"; }
have() { command -v "$1" >/dev/null 2>&1; }

echo "== OS と資源 =="
line "OS" "$(uname -sm)"
if have nproc; then line "CPU のコア数" "$(nproc)"; else line "CPU のコア数" "$(sysctl -n hw.ncpu 2>/dev/null || echo ?)"; fi
if have free; then line "メモリ" "$(free -h | awk '/^Mem:/ {print $2 " (空き " $7 ")"}')"; fi
line "ホームの空き" "$(df -h "${HOME}" | awk 'NR==2 {print $4 " / " $2}')"

echo "== GPU =="
if have nvidia-smi; then
  nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv,noheader
  line "CUDA_VISIBLE_DEVICES" "${CUDA_VISIBLE_DEVICES:-（未設定）}"
else
  echo "nvidia-smi なし（GPU の無いノード、または Mac）"
fi

echo "== 道具（あるかどうかだけ） =="
for t in git uv tmux screen docker podman apptainer singularity sbatch squeue module; do
  if have "${t}"; then line "${t}" "あり（$(command -v "${t}")）"; else line "${t}" "なし"; fi
done
if have docker; then
  if docker info >/dev/null 2>&1; then
    line "docker の権限" "使える（docker グループ。共有サーバーでは root と同じ強さなので扱いは研究室の決まりに従う）"
  else
    line "docker の権限" "コマンドはあるが使えない（デーモンが止まっている、または権限なし。管理者に聞く）"
  fi
fi
if have sbatch; then
  line "Slurm" "あり：ジョブはログインノードで直接走らせず sbatch で出す"
  sinfo -s 2>/dev/null | head -n 8
fi
if have quota; then quota -s 2>/dev/null | tail -n +1 | head -n 6; fi
echo "== 確かめたら、研究室の決まり（GPU の予約、データの置き場所、Docker の可否）を先輩・指導教員に聞く =="

In [ ]:
# uv の代わり（ノートブック版で足したもの）：「uv run --locked python …」を、このノートブックの python で実行する
import stat, shlex
os.makedirs("bin", exist_ok=True)
with open("bin/uv", "w") as f:
    f.write("#!/bin/sh\n# ノートブック版：本物の uv ではない。run と --locked を読みとばし、python をこのノートブックの python にする\n"
            "while [ \"$1\" = run ] || [ \"$1\" = --locked ]; do shift; done\n"
            f"[ \"$1\" = python ] && shift\nexec {shlex.quote(sys.executable)} \"$@\"\n")
os.chmod("bin/uv", 0o755)
os.environ["PATH"] = os.path.abspath("bin") + os.pathsep + os.environ["PATH"]
print(open("bin/uv").read())

In [ ]:
# 設定と、sh()・ok() を用意する（ノートブック版で足したもの）
import re, json, glob, math, shutil, time
import numpy as np
import pandas as pd
sys.path.insert(0, ".")
IN_COLAB = "google.colab" in sys.modules
# ノートブック版で足したもの：Mac でノートブックのフォルダの上にある Git（教科書のフォルダなど）を見ないため（見に行くと、そこのコミットしていない変更で runlog.py が止まる）
os.environ["GIT_CEILING_DIRECTORIES"] = os.path.dirname(os.path.abspath("."))
CUDA = torch.cuda.is_available()
HAS_SMI = CUDA and shutil.which("nvidia-smi") is not None   # GPU を選ばない Colab にも nvidia-smi だけはあることがあるので、CUDA も見る
GPU_ANY = CUDA or torch.backends.mps.is_available()
MACHINE = "colab" if IN_COLAB else "mac"       # 教科書の MACHINE=lab・mac の代わり（ホスト名は書かない）
N_JOBS = 3 if GPU_ANY else 1                   # 5 節で順に流す種の数（教科書の配列ジョブは --array=0-9%2 で 10 本）
print(f"CUDA {CUDA}、nvidia-smi {'あり' if HAS_SMI else 'なし'}、machine {MACHINE}、5 節の本数 {N_JOBS}")


def sh(cmd, check=True, show=True):
    """ターミナルのコマンド（bash）の代わり。出力をそのまま表示し、(出力, 終了コード) を返す"""
    print("$", cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        if show:
            print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    if check:
        assert rc == 0, f"エラーで終わりました（returncode {rc}）。上の表示を見てください"
    return "".join(lines), rc


def final(d):
    return json.load(open(os.path.join(d, "final.json"), encoding="utf-8"))


N_OK = 0
def ok(cond, msg):
    global N_OK
    assert cond, "★確かめ失敗：" + msg
    N_OK += 1
    print("OK", msg)

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
for f in ("run_bg.sh", "job_seeds.sbatch", "lab_check.sh", "gpu_log.sh"):
    out, rc = sh(f"bash -n {f}", check=False)
    ok(rc == 0 and out == "", f"[手順1] bash -n {f} が何も表示しない（文法の誤りが無い）")

## 1. 何が使えるかを調べる（手順 8 の `bash lab_check.sh`）

研究室のサーバーに初めて入った日に、何も変えず、何も入れずに、CPU のコア数・メモリ・ホームの空き・GPU・道具（git・uv・tmux・Docker・Apptainer・Slurm など）を調べます。Colab で実行すると、Colab のランタイム（Linux）のようすが分かります。

**表示を研究室の外に貼らない**のが教科書の決まりです（ホスト名・ユーザー名は表示しない作りですが、サーバーの構成も研究室の情報です）。

In [ ]:
out_lab, _ = sh("bash lab_check.sh")

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
ok("== OS と資源 ==" in out_lab and "== GPU ==" in out_lab and "== 道具" in out_lab, "lab_check.sh が OS と資源・GPU・道具を表示して終わる")
ok(os.uname().nodename not in out_lab, "表示にホスト名が入っていない")
# 「uv あり」と出るのは、このノートブックの bin/uv（本物の uv ではない）

## 2. 空いている GPU の判定（手順 2・9、`gpu_free.py`）

`gpu_free.py` は `nvidia-smi --query-gpu`（メモリ・GPU-Util）と `--query-compute-apps`（動いているプロセス）を読み、「使用メモリが 500 MiB 以下、GPU-Util が 5% 以下、プロセスが無い」GPU を「空き」とします。`[N/A]`・`[Not Supported]` で読めない GPU は、空きとみなしません。研究室に予約表や Slurm があるときは、そちらが優先です（このスクリプトは目安を出すだけ）。

まず見本の値（5 枚）で、次にこの機械の `nvidia-smi` で実行します（`nvidia-smi` が無い Mac では、そのことを表示して止まります）。

In [ ]:
PYQ = shlex.quote(sys.executable)   # このノートブックの python（教科書の python3 の代わり）
out_d1, _ = sh(f"{PYQ} gpu_free.py --demo --need 1")
out_d3, rc_d3 = sh(f"{PYQ} gpu_free.py --demo --need 3", check=False)
print("終了コード", rc_d3)
out_real, rc_real = sh(f"{PYQ} gpu_free.py --need 1", check=False)
print("終了コード", rc_real)

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
rows = {int(l.split()[0]): l for l in out_d1.splitlines() if re.match(r"\s*\d\s", l)}
ok(rows[0].endswith("使用中") and rows[2].endswith("使用中") and rows[1].endswith("空き") and rows[3].endswith("空き"),
   "[手順2] 見本の 5 枚のうち 0 番（23510 MiB・98%・プロセス 1）と 2 番（812 MiB・プロセス 1）が使用中、1 番と 3 番が空き")
ok(rows[4].endswith("不明（使用中扱い）"), "見本の 4 番（[N/A]・[Not Supported]）は「不明（使用中扱い）」")
ok("export CUDA_DEVICE_ORDER=PCI_BUS_ID" in out_d1 and "export CUDA_VISIBLE_DEVICES=1" in out_d1,
   "[手順2] --need 1 で export CUDA_DEVICE_ORDER=PCI_BUS_ID と export CUDA_VISIBLE_DEVICES=1 が表示される")
ok(rc_d3 == 1 and "空きが 3 枚ありません" in out_d3, "III-7-7：--need 3 では「空きが 3 枚ありません」で終了コード 1")
if HAS_SMI:
    print("この機械の GPU：", "空きあり" if rc_real == 0 else "空きなし（ほかの計算が動いている）")
else:
    ok(rc_real == 1 and "nvidia-smi がありません" in out_real, "III-7-7：nvidia-smi の無い機械で --demo を付けないと、そのことを表示して止まる")

## 3. `CUDA_VISIBLE_DEVICES` で GPU の見え方を変える（手順 2・9、`show_devices.py`）

`CUDA_VISIBLE_DEVICES=2` で起動すると、プロセスの中では `cuda:0` が物理的な GPU 2 になります。並べた順に `cuda:0`、`cuda:1`、… と番号が振り直され、**存在しない番号があると、そこから後ろは見えなくなります**（CUDA の決まり。公式ドキュメントで要確認）。空文字列だと 1 枚も見えません。

Colab の GPU は 1 枚（物理的な 0 番）なので、`0`・`""`・`9`・`0,9`・`9,0` で確かめます。研究室のサーバーで「2 と 3 が空いている」ときの `3,2` や `2,9,3` は、確認問題 2 として下のセルで答え合わせします。Mac（MPS）では `CUDA_VISIBLE_DEVICES` は効かず、値を表示するだけです。

In [ ]:
outs = {}
for v in [None, "0", "", "9", "0,9", "9,0"]:
    pre = "env -u CUDA_VISIBLE_DEVICES" if v is None else f"CUDA_VISIBLE_DEVICES='{v}'"
    outs[v], _ = sh(f"{pre} uv run --locked python show_devices.py")
    print()

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
if CUDA:
    n_all = torch.cuda.device_count()
    see = lambda o: int(m.group(1)) if (m := re.search(r"見えている GPU (\d+) 枚", o)) else 0
    ok(see(outs[None]) == n_all and see(outs["0"]) == 1, f"[手順9] 未設定なら全部（{n_all} 枚）、CUDA_VISIBLE_DEVICES=0 なら 1 枚だけ見える")
    ok(see(outs[""]) == 0 and see(outs["9"]) == 0, "空文字列や存在しない番号だけなら、GPU は 1 枚も見えない（torch.cuda.is_available() が False）")
    ok(see(outs["0,9"]) == 1 and see(outs["9,0"]) == 0, "存在しない番号（9）から後ろは見えない：0,9 なら 1 枚、9,0 なら 0 枚")
else:
    ok(all("CUDA なし" in o or "CUDA も MPS もありません" in o for o in outs.values()), "III-7-7：CUDA の無い機械でも show_devices.py は落ちずに表示する")
    ok("CUDA_VISIBLE_DEVICES = 0" in outs["0"], "CUDA_VISIBLE_DEVICES=0 を付けても落ちずに、その値を表示する（MPS には効かない）")


# 確認問題 2：物理的な GPU が 0〜3 の 4 枚のとき、プロセスの中の cuda:i は物理的な何番か（CUDA の決まりを書いた小さな関数）
def visible(cvd, n_phys=4):
    out = []
    for t in cvd.split(","):
        if not t.strip().isdigit() or int(t) >= n_phys:
            break                      # 存在しない番号から後ろは見えない
        out.append(int(t))
    return out


ok(visible("3,2") == [3, 2], "確認問題2：CUDA_VISIBLE_DEVICES=3,2 なら cuda:0 が物理的な 3 番、cuda:1 が 2 番")
ok(visible("2,9,3") == [2], "確認問題2：2,9,3（9 は存在しない）なら、見えるのは物理的な 2 番（cuda:0）だけ")

## 4. `nohup` で裏で動かす（手順 4、`run_bg.sh`）

`run_bg.sh` は、コマンドを `nohup … &` で裏で実行し、出力を `runs/logs/<名前>.log`、プロセス番号を `runs/logs/<名前>.pid` に残します。端末（SSH）を閉じても止まりません。同じ名前で、まだ動いている間にもう一度実行すると「もう動いています」で止まります。

ここでは III-4 の種 0 を 1 本裏で流し、動いている間に同じ名前でもう一度実行し、終わるまで `tail` のかわりにログを読みます。

In [ ]:
shutil.rmtree(f"runs/lab_seed_{MACHINE}", ignore_errors=True)
for p in glob.glob(f"runs/logs/{MACHINE}_seed000.*"):
    os.remove(p)
NAME = f"{MACHINE}_seed000"
CMD = f"uv run --locked python train_repro.py --config configs/seed_small.yaml --seed 0 --out runs/lab_seed_{MACHINE} --machine {MACHINE}"
out_bg, _ = sh(f"bash run_bg.sh {NAME} -- {CMD}")
out_bg2, rc_bg2 = sh(f"bash run_bg.sh {NAME} -- {CMD}", check=False)     # 動いている間にもう一度
pid = int(open(f"runs/logs/{NAME}.pid").read())
while subprocess.run(["kill", "-0", str(pid)], capture_output=True).returncode == 0:   # 終わるまで待つ（端末を閉じた代わり）
    time.sleep(2)
log = open(f"runs/logs/{NAME}.log", encoding="utf-8").read()
print(log)

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
ok(out_bg.startswith("開始しました：PID") and "様子を見る：tail -f" in out_bg, "[手順4] 「開始しました：PID …」と表示してすぐ戻る")
ok(rc_bg2 == 1 and "もう動いています" in out_bg2, "[手順4] 動いている間に同じ名前で実行すると「もう動いています」で止まる")
ok(log.startswith("# 開始") and "# CUDA_VISIBLE_DEVICES=" in log and '"test_acc"' in log.splitlines()[-1],
   "ログの先頭に開始の時刻・CUDA_VISIBLE_DEVICES・コマンド、最後に III-4 と同じ {\"val_acc\": …, \"test_acc\": …} が書かれる")
r0 = final(f"runs/lab_seed_{MACHINE}/seed000")
ok(r0["machine"] == MACHINE, f"[手順4] runs/lab_seed_{MACHINE}/seed000/final.json ができた（test_acc {r0['test_acc']}、wall_s {r0['wall_s']}。筆者の Mac の MPS では val_acc 0.895・test_acc 0.916）")

## 5. ジョブスクリプトを Slurm なしで動かす（手順 5・10、`job_seeds.sbatch`）

`#SBATCH` の行は Slurm だけが読み、bash にとってはただのコメントです。なので `SLURM_ARRAY_TASK_ID` を自分で付ければ、Slurm の無い機械でも 1 本だけ試せます。

1. `DRY=1` で、実行せずにコマンドだけ表示する（III-4 の `run_seeds.sh` と同じ形になっているか）
2. `WORKERS=2` を付けると `--set train.workers=2` が足される。設定のハッシュ（config_hash）が変わるので、比べる実行どうしで同じ値にそろえる
3. 実際に 1 本動かし、もう一度実行すると「final.json があるので、とばします」

最後に、配列ジョブ（`--array=0-9%2`）のまねとして、`SLURM_ARRAY_TASK_ID` を 0、1、2 と変えて 1 本ずつ流し、III-4 の `seed_stats.py` で集計します（手順 10）。

In [ ]:
OUT = f"runs/lab_seed_{MACHINE}"
ENV = f"OUT_ROOT={OUT} MACHINE={MACHINE}"
out_dry, _ = sh(f"DRY=1 SLURM_ARRAY_TASK_ID=1 {ENV} bash job_seeds.sbatch")
out_dryw, _ = sh(f"DRY=1 WORKERS=2 SLURM_ARRAY_TASK_ID=1 {ENV} bash job_seeds.sbatch")
out_j1, _ = sh(f"SLURM_ARRAY_TASK_ID=1 {ENV} bash job_seeds.sbatch")
out_j1b, _ = sh(f"SLURM_ARRAY_TASK_ID=1 {ENV} bash job_seeds.sbatch")      # 2 回目はとばす

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
ok("ジョブ local 配列 local_1 種 1" in out_dry, "III-7-7：Slurm なしでは「ジョブ local 配列 local_1 種 1」と表示する")
ok("train_repro.py --config configs/seed_small.yaml --seed 1 --out " + OUT in out_dry and "--set" not in out_dry,
   f"[手順5] 表示されたコマンドは run_seeds.sh と同じ形（--seed 1 --out {OUT}）")
ok("--set train.workers=2" in out_dryw, "WORKERS=2 を付けると --set train.workers=2 が付く（表示だけ）")
ok(os.path.exists(f"{OUT}/seed001/final.json") and "final.json があるので、とばします" in out_j1b,
   f"[手順5] {OUT}/seed001/ に final.json ができ、2 回目は「final.json があるので、とばします」")
from runlog import load_config, config_hash
h0 = config_hash(load_config("configs/seed_small.yaml"))
h2 = config_hash(load_config("configs/seed_small.yaml", ["train.workers=2"]))
ok(h0 == "ad7976cfec9d" and h2 == "4f34bd35d6f6" and final(f"{OUT}/seed001")["config_hash"] == h0,
   f"III-7-7：WORKERS=2 を付けると config_hash が {h0} から {h2} に変わる（どの機械でも同じ値）")

In [ ]:
# 配列ジョブのまね：Slurm なら sbatch job_seeds.sbatch（--array=0-9%2）で 10 本が 2 本ずつ並んで流れる
for task in range(N_JOBS):
    sh(f"SLURM_ARRAY_TASK_ID={task} {ENV} bash job_seeds.sbatch", show=False)
    print(f"種 {task}：test_acc {final(f'{OUT}/seed{task:03d}')['test_acc']}")
if N_JOBS >= 2:
    out_st, _ = sh(f"uv run --locked python seed_stats.py {OUT} --out {OUT}.png")
# 本番（教科書の手順10）：研究室の Slurm で mkdir -p runs/logs && sbatch job_seeds.sbatch → sacct で 10 本 COMPLETED

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
done = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(f"{OUT}/seed*/final.json"))
ok(done == sorted({f"seed{t:03d}" for t in range(N_JOBS)} | {"seed000", "seed001"}),
   f"[手順10] 流した種の final.json がそろう（{', '.join(done)}。4 節の種 0 と、手で流した種 1 はとばされる）")
ok(final(f"{OUT}/seed000")["state_sha"] != "" and all(final(f"{OUT}/{d}")["machine"] == MACHINE for d in done),
   "run_record.json・final.json の machine は自分で付けた名前（ホスト名ではない）")

## 6. 1 枚に絞って GPU-Util を記録する（手順 9、III-5 の `gpu_log.sh`）

研究室のサーバーでは、`gpu_free.py --need 1` が表示した 2 行の `export` を（tmux の中で）打ち、`show_devices.py` で 1 枚だけ見えることを確かめてから、`gpu_log.sh` で学習を 1 本流します。`gpu_log.sh` は `CUDA_VISIBLE_DEVICES` があればその GPU だけを記録し、まとめもその GPU の 1 行だけになります（`nvidia-smi` そのものは `CUDA_VISIBLE_DEVICES` を見ずに全部の GPU を表示するため）。

Colab では GPU は 1 枚ですが、同じ手順で `CUDA_VISIBLE_DEVICES=0` にして記録します。`nvidia-smi` が無い Mac では、記録せずにコマンドだけ実行します。

In [ ]:
shutil.rmtree(f"runs/lab_seed_gpulog", ignore_errors=True)
pre = "export CUDA_DEVICE_ORDER=PCI_BUS_ID; export CUDA_VISIBLE_DEVICES=0; " if HAS_SMI else ""
out_gl, _ = sh(pre + "GPU_LOG_DIR=runs/gpu_log bash gpu_log.sh lab_seed000 -- uv run --locked python train_repro.py "
               f"--config configs/seed_small.yaml --seed 0 --out runs/lab_seed_gpulog --machine {MACHINE}")

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
r_gl = final("runs/lab_seed_gpulog/seed000")
ok(r_gl["test_acc"] == r0["test_acc"] or not CUDA, "同じ種・同じ device の学習は、4 節と同じ test_acc（決定的）")
if HAS_SMI:
    ok("記録する GPU：CUDA_VISIBLE_DEVICES=0" in out_gl, "[手順9] gpu_log.sh が「記録する GPU：CUDA_VISIBLE_DEVICES=…」と表示する")
    g = pd.read_csv("runs/gpu_log/lab_seed000.csv", skipinitialspace=True)
    ucol = [c for c in g.columns if c.startswith("utilization.gpu")][0]
    util = pd.to_numeric(g[ucol].astype(str).str.replace("%", "").str.strip(), errors="coerce")
    ok(g[[c for c in g.columns if c.startswith("index")][0]].nunique() == 1, "[手順9] 記録は選んだ GPU の 1 枚だけ")
    print(f"GPU-Util の平均 {util.mean():.0f}%（{len(util)} 秒分）。記録する")
else:
    ok("nvidia-smi がありません" in out_gl, "nvidia-smi の無い機械では、記録せずにコマンドだけ実行する")
print("""確認問題7：共有サーバーで GPU-Util の平均が 25% だったら、GPU は 4 分の 3 の時間データを待っている（CPU での合成・読み込み、
ワーカー数、CPU のコア数）。その GPU をほかの人は使えないのに、計算はほとんどしていない。最初に試すのは III-5 の切り分け
（データ待ちの割合を測り、ワーカー数を --cpus-per-task 以下で増やす・前処理を保存する）。この小さなモデルでは、Colab でも
GPU-Util が低めに出ることが多い""")

## 7. 確認問題 3：配列ジョブの時間と GPU 時間

種 0〜19 の 20 本、1 本 15 分を `--array=0-19%4`（同時に 4 本まで）で出したとき、待ち時間を除いて全部終わるまで何分か、GPU を使う量は合計何 GPU 時間か。`%4` を `%2` にするとどう変わるか。

**先に紙で解いてから**、下のセルで答え合わせします。小さな「同時に k 本まで」のスケジューラで、終わった順に次のジョブを始めるようすを計算します。

In [ ]:
def array_job(n, minutes, k):
    """同時に k 本まで。空いたところに次のジョブを入れる。全部終わる時刻 [分] と GPU 時間 [時] を返す"""
    slots = [0.0] * k
    for _ in range(n):
        i = min(range(k), key=lambda j: slots[j])
        slots[i] += minutes
    return max(slots), n * minutes / 60


for k in (4, 2):
    t, gh = array_job(20, 15, k)
    print(f"--array=0-19%{k}：全部終わるまで {t:.0f} 分、GPU 時間 {gh:.1f} 時間（1 枚ずつ使う）")

In [ ]:
# 確かめ（教科書 III-7-5 の「確かめること」と III-7-7 の結果の例・確認問題のうち、機械に依らず成り立つことだけ）
ok(array_job(20, 15, 4) == (75, 5.0) and array_job(20, 15, 2) == (150, 5.0),
   "確認問題3：%4 なら 20/4 = 5 回 × 15 分 = 75 分、%2 なら 150 分。GPU 時間はどちらも 20 × 15 分 = 5 時間（同時の数は総量を変えない）")
ok(math.ceil(10 / 2) * array_job(1, 1, 1)[0] == 5, "教科書の job_seeds.sbatch（--array=0-9%2）は、1 本の時間の 5 倍で終わる")
print(f"ok() の数：{N_OK}")

## おわりに

- `gpu_free.py` で空いている GPU を選び、`CUDA_VISIBLE_DEVICES` で自分のプロセスに見える GPU を 1 枚に絞り、`show_devices.py` で確かめてから学習を始める、という順番を練習しました。存在しない番号から後ろは見えなくなること、空文字列だと 1 枚も見えないことも確かめました。
- `run_bg.sh`（nohup）とジョブスクリプトは、どちらも「終わった種はとばす」ので、止まっても同じコマンドで続きから流せます。
- 研究室のサーバーで実際に動かすのは、**研究室の決まり（手順 6 の 6 つ）を聞いてから**です。研究データの置き場所と持ち出してよい範囲、GPU の使い方（予約表・Slurm・声かけ）、Docker・Apptainer・uv のどれを使うかを、先に実験ノートに書いてください。